# Image Inspection

Inspect 232 training images one by one and flag problematic ones.
Record findings in the `FLAGGED` list at the bottom.

Checklist:
1. Dark or blurry
2. Reflection / shadow
3. Pill clipped at edge
4. Unusual background
5. Strong rotation
6. Overlapping objects
7. Bbox mismatch
8. Suspected label error
9. Large appearance gap within same class
10. Similar appearance across different classes

## 1. Download Dataset

Skips download if data already exists locally.

In [ ]:
# !uv add kagglehub

import json
from pathlib import Path

# Kaggle auth — write ~/.kaggle/kaggle.json
try:
    from google.colab import userdata
    kaggle_username = userdata.get("KAGGLE_USERNAME")
    kaggle_key = userdata.get("KAGGLE_KEY")
except ImportError:
    from getpass import getpass
    kaggle_username = input("Kaggle Username: ")
    kaggle_key = getpass("Kaggle API Key: ")

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
kaggle_json_path = kaggle_dir / "kaggle.json"
kaggle_json_path.write_text(json.dumps({"username": kaggle_username, "key": kaggle_key}))
kaggle_json_path.chmod(0o600)
print("Kaggle auth done")

# Locate data/raw
current_path = Path.cwd().resolve()

for path_candidate in [current_path, *current_path.parents]:
    raw_root_path = path_candidate / "data" / "raw"
    if raw_root_path.exists():
        break
else:
    raw_root_path = current_path / "data" / "raw"
    raw_root_path.mkdir(parents=True, exist_ok=True)

raw_data_path = raw_root_path / "ai11-level1-project"
dataset_path = raw_data_path / "sprint_ai_project1_data"
required_data_dirs = [
    dataset_path / "train_images",
    dataset_path / "test_images",
    dataset_path / "train_annotations",
]

if all(d.exists() for d in required_data_dirs):
    print("Dataset already exists. Skip download.")
else:
    import kagglehub
    raw_data_path.mkdir(parents=True, exist_ok=True)
    kagglehub.competition_download("ai11-level1-project", output_dir=str(raw_data_path))

print("Dataset path:", dataset_path)

## 2. Load Annotations

In [ ]:
import matplotlib.patches as patches
import matplotlib.pyplot as plt
from PIL import Image

train_image_path = dataset_path / "train_images"
annotation_path = dataset_path / "train_annotations"

annotation_paths = sorted(annotation_path.rglob("*.json"))
bbox_by_image = {}
category_by_id = {}

for annotation_file_path in annotation_paths:
    with annotation_file_path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    for category in data.get("categories", []):
        category_by_id[category["id"]] = category["name"]

    image_id_to_file_name = {img["id"]: img["file_name"] for img in data.get("images", [])}

    for ann in data.get("annotations", []):
        file_name = image_id_to_file_name[ann["image_id"]]
        bbox_by_image.setdefault(file_name, []).append({
            "bbox": ann["bbox"],
            "category_id": ann["category_id"],
            "category_name": category_by_id.get(ann["category_id"], "unknown"),
        })

train_image_files = sorted(train_image_path.glob("*.png"))
print(f"Total training images: {len(train_image_files)}")
print(f"Annotation files: {len(annotation_paths)}")

## 3. Grid View

Shows 16 images at a time. Change `START_IDX` to see the next batch.

- `START_IDX = 0` → images 0–15
- `START_IDX = 16` → images 16–31
- `START_IDX = 32` → images 32–47
- ... and so on up to 231

Note the index number shown in each title (e.g. `[5]`), then use it in the Detail View below.

In [ ]:
START_IDX = 0  # Change to 16, 32, 48 ... to see the next batch
N_COLS = 4
N_ROWS = 4

batch_files = train_image_files[START_IDX : START_IDX + N_COLS * N_ROWS]
fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(20, 20))
axes = axes.flatten()

for ax, image_file in zip(axes, batch_files):
    image = Image.open(image_file).convert("RGB")
    ax.imshow(image)

    for ann in bbox_by_image.get(image_file.name, []):
        x, y, w, h = ann["bbox"]
        rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor="lime", facecolor="none")
        ax.add_patch(rect)

    idx = train_image_files.index(image_file)
    n_bbox = len(bbox_by_image.get(image_file.name, []))
    ax.set_title(f"[{idx}] bbox:{n_bbox}\n{image_file.name[:28]}", fontsize=7)
    ax.axis("off")

for ax in axes[len(batch_files):]:
    ax.axis("off")

plt.suptitle(f"Images [{START_IDX} - {START_IDX + len(batch_files) - 1}]", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Detail View

Set `TARGET_IDX` to the image index you want to inspect closely.

In [ ]:
TARGET_IDX = 0  # Change to the index number from the grid view

target_file = train_image_files[TARGET_IDX]
image = Image.open(target_file).convert("RGB")
bboxes = bbox_by_image.get(target_file.name, [])

fig, ax = plt.subplots(figsize=(10, 13))
ax.imshow(image)

colors = ["lime", "red", "cyan", "yellow"]
for i, ann in enumerate(bboxes):
    x, y, w, h = ann["bbox"]
    color = colors[i % len(colors)]
    rect = patches.Rectangle((x, y), w, h, linewidth=3, edgecolor=color, facecolor="none")
    ax.add_patch(rect)
    ax.text(
        x, max(y - 5, 0),
        ann["category_name"],
        color=color,
        fontsize=9,
        bbox={"facecolor": "black", "alpha": 0.6, "pad": 2},
    )

ax.set_title(f"[{TARGET_IDX}] {target_file.name}", fontsize=11)
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"File: {target_file.name}")
print(f"Bboxes: {len(bboxes)}")
for i, ann in enumerate(bboxes):
    print(f"  [{i}] {ann['category_name']} — bbox: {ann['bbox']}")

## 5. Flag Problematic Images

Add entries to `FLAGGED` as you inspect.
When done, copy findings to `experiment/image_inspection.md`.

In [ ]:
CHECKLIST = [
    "Dark or blurry",
    "Reflection / shadow",
    "Pill clipped at edge",
    "Unusual background",
    "Strong rotation",
    "Overlapping objects",
    "Bbox mismatch",
    "Suspected label error",
    "Large appearance gap within same class",
    "Similar appearance across different classes",
]

FLAGGED = [
    # Format: {"idx": image_index, "file": "filename.png", "issues": [issue_numbers], "memo": "note"}
    # Example:
    # {"idx": 5, "file": "K-xxx.png", "issues": [1, 3], "memo": "very dark, right pill clipped"},
]

print(f"Flagged images: {len(FLAGGED)}")
for item in FLAGGED:
    issues_str = ", ".join([CHECKLIST[i - 1] for i in item["issues"]])
    print(f"  [{item['idx']}] {item['file']}")
    print(f"       issues: {issues_str}")
    print(f"       memo  : {item['memo']}")